# ББСБ асуулт-хариулт — Colab (үнэгүй T4)

1. Google Drive-д **MyDrive/t2s/** хавтас үүсгээд дотор нь хуулна: `t2s_code.zip`, `t2s_adapter_d2.zip.part00`, `part01`, … ба `SHA256.txt`.
2. Colab: **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all**. Drive-д нэвтрэх зөвшөөрөл асууна — зөвшөөрнө.
4. ~25–35 минутын дараа сүүлийн нүдэнд **`WEB URL: https://….trycloudflare.com/?k=…`** гарна. Тэр холбоосыг (бүтнээр нь, `?k=` хэсэгтэй) утас эсвэл компьютерийн браузерт нээгээд асууна. «FLOW BUILDER:» холбоосоор pipeline-ийг node-оор угсарч турших.
5. Энэ Colab-ийн табыг хаавал вэб зогсоно. Үнэгүй Colab хэдэн цагийн дараа өөрөө тасардаг.

In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
SRC = Path("/content/drive/MyDrive/t2s")
assert (SRC / "t2s_code.zip").exists(), "MyDrive/t2s/t2s_code.zip алга"
assert list(SRC.glob("t2s_adapter_d2.zip.part*")), "MyDrive/t2s/t2s_adapter_d2.zip.part00 … алга"
REPO = Path("/content/t2s")
shutil.rmtree(REPO, ignore_errors=True)
zipfile.ZipFile(SRC / "t2s_code.zip").extractall(REPO)
os.chdir(REPO)
os.environ.update(T2S_PG_PASSWORD="t2s_nb_admin", VLLM_USE_FLASHINFER_SAMPLER="0", ADAPTER_SRC=str(SRC),
                  STAGE="web", WEB_MIN="300", HF_HUB_ENABLE_HF_TRANSFER="0")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

def run(cmd):
    p = subprocess.Popen(["bash", "-c", f"source scripts/server/env.sh && {cmd}"], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")

In [ ]:
# PostgreSQL 16, vLLM, DB (seed 42) — ~10–15 мин
run("bash scripts/notebook/bootstrap.sh")

In [ ]:
# D2 нэгтгэх + 2 загвар (4-bit, нэг T4) + вэб + tunnel. Доор «WEB URL:» мөр гарна (~10–20 мин).
run("bash scripts/server/run_plan_l4.sh")